# Step-back

구체 질문과 한 단계 넓힌 질문의 결과를 함께 사용합니다.

## 실행 환경 준비

고정한 Python 패키지를 확인합니다.

In [1]:
%pip install -q langchain==1.4.0 langchain-openai==1.6.1 python-dotenv==1.2.3 numpy==2.5.3
import sys
print("준비 완료:", sys.version.split()[0])

Note: you may need to restart the kernel to use updated packages.
준비 완료: 3.12.13


## .env 찾기

가까운 .env에서 키를 읽고 존재 여부만 표시합니다.

In [2]:
# 현재 폴더에서 가까운 .env를 찾되 비밀 값은 출력하지 않습니다.
import os, time
from dotenv import find_dotenv, load_dotenv
env_path=find_dotenv(usecwd=True)
load_dotenv(env_path)
print("OPENAI_API_KEY:","있음" if os.getenv("OPENAI_API_KEY") else "없음")

OPENAI_API_KEY: 있음


## 공통 데이터

Ch12의 축소 데이터와 Ch11 방식의 정답 문서 ID를 사용합니다.

In [3]:
# 배포 묶음에서도 같은 상대 경로를 찾도록 프로젝트 루트를 거슬러 올라갑니다.
import json
from pathlib import Path
import numpy as np
root=next(p for p in [Path.cwd(),*Path.cwd().parents] if (p/"labs").is_dir())
data_dir=root/"labs/part4/ch14-rag-patterns/data"
docs=json.loads((data_dir/"documents.json").read_text(encoding="utf-8"))
queries=json.loads((data_dir/"queries.json").read_text(encoding="utf-8"))
print("문서·질문:",len(docs),len(queries))

문서·질문: 12 8


## 모델과 임베딩

모델과 256차원 문서 벡터를 준비합니다.

In [4]:
# 256차원은 Ch12와 같은 비교 조건이며 행 하나가 문서 하나입니다.
from langchain.chat_models import init_chat_model
from langchain_openai import OpenAIEmbeddings
chat=init_chat_model("gpt-4.1-mini",model_provider="openai",temperature=0)
embedder=OpenAIEmbeddings(model="text-embedding-3-small",dimensions=256)
vectors=np.array(embedder.embed_documents([d["text"] for d in docs]))
print("임베딩 배열:",vectors.shape)

임베딩 배열: (12, 256)


## 검색·생성 함수

검색 점수, 토큰, 시간, 추정 비용을 같은 방식으로 기록합니다.

In [5]:
# 코사인 유사도는 내적을 두 벡터의 길이로 나누어 계산합니다.
from typing import Any
usage_log: list[dict[str, float | int]] = []

def search(question: str, k: int = 3) -> list[tuple[dict[str, Any], float]]:
    q=np.array(embedder.embed_query(question))
    scores=vectors@q/(np.linalg.norm(vectors,axis=1)*np.linalg.norm(q))
    # argsort 결과를 뒤집어 점수가 큰 문서부터 k개를 고릅니다.
    return [(docs[i],float(scores[i])) for i in np.argsort(scores)[::-1][:k]]
def ask(prompt: str) -> tuple[str, int, float]:
    started=time.perf_counter()
    msg=chat.invoke(prompt)
    usage=msg.usage_metadata or {}
    details=usage.get("input_token_details", {})
    input_tokens=usage.get("input_tokens", 0)
    cached_tokens=details.get("cache_read", 0)
    output_tokens=usage.get("output_tokens", 0)
    tokens=usage.get("total_tokens", 0)
    # 2026-09-10 gpt-4.1-mini 공개 단가로 추정하며 실제 청구액과 다를 수 있습니다.
    usd=((input_tokens-cached_tokens)*0.40+cached_tokens*0.10+output_tokens*1.60)/1_000_000
    usage_log.append({"input":input_tokens,"cached":cached_tokens,"output":output_tokens,"usd":usd})
    return msg.text,tokens,round((time.perf_counter()-started)*1000,1)

def show_hits(hits: list[tuple[dict[str, Any], float]]) -> list[tuple[str, float]]:
    return [(d["doc_id"],round(score,3)) for d,score in hits]

## 구체 질문 검색

날짜와 시각을 포함한 질문으로 먼저 검색합니다. FAC-003의 순위와 유사도를 기준선으로 기록합니다.

In [6]:
# 날짜와 시각을 포함한 원 질문으로 먼저 운영 규정을 검색합니다.
question="토요일 오후 6시에 자료실에 가도 되나요?"
specific=search(question,k=2)
print("구체 질문:",show_hits(specific))

구체 질문: [('FAC-003', 0.561), ('ACC-002', 0.309)]


## 넓은 질문 생성

세부 질문을 일반 운영 규칙 질문으로 바꾸고 다시 검색합니다. 생성 토큰·시간과 넓은 검색 점수를 확인합니다.

In [7]:
# 세부 시각에서 한 단계 물러나 시설 운영 규칙을 묻습니다.
step_back,tokens,latency=ask(f"세부 시각 판단에 필요한 일반 규칙을 찾도록 더 넓은 검색 질문 하나를 쓰세요:{question}")
broad=search(step_back,k=2)
print("Step-back:",step_back)
print("넓은 검색:",show_hits(broad),"토큰:",tokens,"ms:",latency)

Step-back: "공공시설 이용 가능 시간과 요일별 운영 규정에 관한 일반적인 안내"
넓은 검색: [('FAC-003', 0.53), ('LIB-001', 0.4)] 토큰: 65 ms: 1072.2


## 두 문맥으로 답하기

두 검색의 문서를 ID로 합쳐 최종 답을 생성합니다. 폐관 시각 판단과 두 번째 생성 호출의 비용을 확인합니다.

In [8]:
# 원 질문과 넓은 질문의 문서를 ID로 합쳐 답변 근거를 만듭니다.
unique={d["doc_id"]:d for d,_ in specific+broad}
context="\n".join(f"[{k}] {v['text']}" for k,v in unique.items())
answer,tokens,latency=ask(f"근거로만 답하세요.\n{context}\n질문:{question}")
print(answer,"\n토큰·ms:",tokens,latency)

[FAC-003]에 따르면 토요일은 오후 5시에 문을 닫으므로, 토요일 오후 6시에 자료실에 가는 것은 불가능합니다. 
토큰·ms: 166 1984.3
